# Tungsten Automation ML Models

Trains and registers 4 ML models in the Snowflake Model Registry:
1. **Customer Churn Risk** - Binary classification (RandomForest)
2. **Extraction Accuracy Prediction** - Regression (GradientBoosting)
3. **Invoice Processing Anomaly Detection** - IsolationForest
4. **Platform Capacity Forecasting** - Regression (GradientBoosting)

**Prerequisites:** Run SQL files 01-03b before this notebook.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f'Connected: {session.get_current_database()}.{session.get_current_schema()}')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, IsolationForest, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, r2_score
from snowflake.ml.registry import Registry

## Model 1: Customer Churn Risk

In [ ]:
SELECT c.CUSTOMER_ID,
       AVG(u.DOCUMENTS_PROCESSED) AS AVG_DOCS,
       (AVG(CASE WHEN u.USAGE_DATE > DATEADD('day',-30,CURRENT_DATE()) THEN u.DOCUMENTS_PROCESSED END) -
        AVG(CASE WHEN u.USAGE_DATE BETWEEN DATEADD('day',-60,CURRENT_DATE()) AND DATEADD('day',-30,CURRENT_DATE()) THEN u.DOCUMENTS_PROCESSED END)) /
       NULLIF(AVG(CASE WHEN u.USAGE_DATE BETWEEN DATEADD('day',-60,CURRENT_DATE()) AND DATEADD('day',-30,CURRENT_DATE()) THEN u.DOCUMENTS_PROCESSED END),0) AS USAGE_TREND,
       COALESCE(t.TICKET_COUNT,0) AS TICKET_COUNT,
       COALESCE(t.AVG_CSAT,3.5) AS AVG_CSAT,
       COALESCE(s.DAYS_TO_RENEWAL,180) AS DAYS_TO_RENEWAL,
       CASE WHEN (AVG(CASE WHEN u.USAGE_DATE > DATEADD('day',-30,CURRENT_DATE()) THEN u.DOCUMENTS_PROCESSED END) -
                  AVG(CASE WHEN u.USAGE_DATE BETWEEN DATEADD('day',-60,CURRENT_DATE()) AND DATEADD('day',-30,CURRENT_DATE()) THEN u.DOCUMENTS_PROCESSED END)) /
                 NULLIF(AVG(CASE WHEN u.USAGE_DATE BETWEEN DATEADD('day',-60,CURRENT_DATE()) AND DATEADD('day',-30,CURRENT_DATE()) THEN u.DOCUMENTS_PROCESSED END),0) < -0.15
            AND COALESCE(t.AVG_CSAT,3.5) < 3.0 THEN 1 ELSE 0 END AS CHURNED
FROM TA_INTELLIGENCE.RAW.CUSTOMER_DIM c
JOIN TA_INTELLIGENCE.RAW.CUSTOMER_USAGE_METRICS u ON c.CUSTOMER_ID = u.CUSTOMER_ID
LEFT JOIN (SELECT CUSTOMER_ID, COUNT(*) AS TICKET_COUNT, AVG(CSAT_SCORE) AS AVG_CSAT FROM TA_INTELLIGENCE.RAW.SUPPORT_TICKETS WHERE OPENED_TS > DATEADD('day',-90,CURRENT_DATE()) GROUP BY CUSTOMER_ID) t ON c.CUSTOMER_ID = t.CUSTOMER_ID
LEFT JOIN (SELECT CUSTOMER_ID, MIN(DATEDIFF('day',CURRENT_DATE(),RENEWAL_DATE)) AS DAYS_TO_RENEWAL FROM TA_INTELLIGENCE.RAW.CUSTOMER_SUBSCRIPTIONS WHERE STATUS='ACTIVE' GROUP BY CUSTOMER_ID) s ON c.CUSTOMER_ID = s.CUSTOMER_ID
WHERE u.USAGE_DATE > DATEADD('day',-90,CURRENT_DATE())
GROUP BY c.CUSTOMER_ID, t.TICKET_COUNT, t.AVG_CSAT, s.DAYS_TO_RENEWAL

In [ ]:
# Train churn model
df_c = df_churn.dropna()
features = ['AVG_DOCS','USAGE_TREND','TICKET_COUNT','AVG_CSAT','DAYS_TO_RENEWAL']
X = df_c[features]; y = df_c['CHURNED']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

churn_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
churn_model.fit(X_train, y_train)
y_pred = churn_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, zero_division=0)
print(f'Churn Model - Accuracy: {acc:.3f}, F1: {f1:.3f}')

In [ ]:
# Register churn model
registry = Registry(session=session, database_name='TA_INTELLIGENCE', schema_name='ANALYTICS')
mv = registry.log_model(churn_model, model_name='CHURN_RISK_MODEL', version_name='V1',
    sample_input_data=X_train, conda_dependencies=['scikit-learn'],
    comment='Customer churn risk classifier (90-day) - RandomForest')
mv.set_metric('accuracy', acc)
mv.set_metric('f1_score', f1)
print('CHURN_RISK_MODEL V1 registered')

## Model 2: Extraction Accuracy Prediction

In [ ]:
SELECT j.DOCUMENT_TYPE, cl.MODEL_VERSION, COUNT(*) AS SAMPLE_COUNT,
       AVG(j.PAGE_COUNT) AS AVG_PAGES,
       AVG(CASE WHEN cl.PREDICTED_CLASS = cl.ACTUAL_CLASS THEN 1.0 ELSE 0.0 END) AS ACCURACY
FROM TA_INTELLIGENCE.RAW.DOCUMENT_PROCESSING_JOBS j
JOIN TA_INTELLIGENCE.RAW.CLASSIFICATION_EVENTS cl ON j.JOB_ID = cl.JOB_ID
WHERE j.RECEIVED_TS > DATEADD('month',-6,CURRENT_TIMESTAMP())
GROUP BY j.DOCUMENT_TYPE, cl.MODEL_VERSION HAVING COUNT(*) > 50

In [ ]:
# Train accuracy model
X_a = df_accuracy[['SAMPLE_COUNT','AVG_PAGES','ACCURACY']].copy()
y_a = (df_accuracy['ACCURACY'] + np.random.normal(0, 0.02, len(df_accuracy))).clip(0,1)
X_tr, X_te, y_tr, y_te = train_test_split(X_a, y_a, test_size=0.2, random_state=42)

acc_model = GradientBoostingRegressor(n_estimators=50, random_state=42, max_depth=3)
acc_model.fit(X_tr, y_tr)
mae = mean_absolute_error(y_te, acc_model.predict(X_te))
r2 = r2_score(y_te, acc_model.predict(X_te))
print(f'Extraction Accuracy Model - MAE: {mae:.4f}, R2: {r2:.3f}')

In [ ]:
mv2 = registry.log_model(acc_model, model_name='EXTRACTION_ACCURACY_MODEL', version_name='V1',
    sample_input_data=X_tr, conda_dependencies=['scikit-learn'],
    comment='Extraction accuracy prediction - GradientBoosting')
mv2.set_metric('mae', mae); mv2.set_metric('r2_score', r2)
print('EXTRACTION_ACCURACY_MODEL V1 registered')

## Model 3: Invoice Anomaly Detection

In [ ]:
SELECT TRANSACTION_ID, AMOUNT,
       DATEDIFF('second', SUBMITTED_TS, COALESCE(DELIVERED_TS, SUBMITTED_TS)) AS DELIVERY_TIME_SEC,
       HOUR(SUBMITTED_TS) AS SUBMIT_HOUR
FROM TA_INTELLIGENCE.RAW.INVOICE_NETWORK_TRANSACTIONS
WHERE SUBMITTED_TS > DATEADD('month',-3,CURRENT_TIMESTAMP()) AND DELIVERED_TS IS NOT NULL
LIMIT 100000

In [ ]:
X_inv = df_inv[['AMOUNT','DELIVERY_TIME_SEC','SUBMIT_HOUR']].dropna()
anom_model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
anom_model.fit(X_inv)
anom_rate = (anom_model.predict(X_inv) == -1).mean()
print(f'Anomaly Model - Rate: {anom_rate:.3f} ({int(anom_rate*len(X_inv))} flagged)')

In [ ]:
mv3 = registry.log_model(anom_model, model_name='INVOICE_ANOMALY_MODEL', version_name='V1',
    sample_input_data=X_inv.head(100), conda_dependencies=['scikit-learn'],
    comment='Invoice anomaly detection - IsolationForest (5% contamination)')
mv3.set_metric('anomaly_rate', float(anom_rate))
print('INVOICE_ANOMALY_MODEL V1 registered')

## Model 4: Capacity Forecasting

In [ ]:
SELECT PRODUCT_ID, REGION, AVG(QUEUE_DEPTH) AS AVG_QUEUE,
       STDDEV(QUEUE_DEPTH) AS STD_QUEUE, MAX(QUEUE_DEPTH) AS MAX_QUEUE,
       AVG(QUEUE_DEPTH) * (1 + UNIFORM(0.0, 0.15, RANDOM())) AS NEXT_PERIOD_QUEUE
FROM TA_INTELLIGENCE.RAW.PLATFORM_HEALTH_METRICS
WHERE METRIC_HOUR > DATEADD('month',-3,CURRENT_TIMESTAMP())
GROUP BY PRODUCT_ID, REGION

In [ ]:
X_c = df_cap[['AVG_QUEUE','STD_QUEUE','MAX_QUEUE']].dropna()
y_c = df_cap['NEXT_PERIOD_QUEUE'].loc[X_c.index]
cap_model = GradientBoostingRegressor(n_estimators=50, random_state=42, max_depth=3)
cap_model.fit(X_c, y_c)
mae_c = mean_absolute_error(y_c, cap_model.predict(X_c))
r2_c = r2_score(y_c, cap_model.predict(X_c))
print(f'Capacity Model - MAE: {mae_c:.1f}, R2: {r2_c:.3f}')

In [ ]:
mv4 = registry.log_model(cap_model, model_name='CAPACITY_FORECAST_MODEL', version_name='V1',
    sample_input_data=X_c, conda_dependencies=['scikit-learn'],
    comment='Platform capacity forecast - GradientBoosting on queue depth')
mv4.set_metric('mae', mae_c); mv4.set_metric('r2_score', r2_c)
print('CAPACITY_FORECAST_MODEL V1 registered')

## Verification

In [ ]:
SHOW MODELS IN SCHEMA TA_INTELLIGENCE.ANALYTICS

In [ ]:
print(f'\nModels registered: {len(df_models)}')
if len(df_models) > 0:
    print(df_models[['name','comment']].to_string(index=False))